# 🎭 Session 4: Decorators, Closures & Functional Patterns
### *Write code that writes code — Python's most elegant feature*

---

| Info | Detail |
|------|--------|
| ⏱️ Duration | ~60 minutes |
| 📝 Exercises | 15 (with hints & solutions) |
| 🎯 Goal | Master closures, decorators, functools, walrus operator |
| 📋 Prerequisites | Sessions 1–3 |

---

## 4.1 Functions Are Objects 🧱

In Python, functions can be **assigned**, **passed**, and **returned** — just like any other object.

In [ ]:
def greet(name):
    return f"Hello, {name}!"

# Assign to variable
say_hi = greet
print(say_hi("Priya"))    # Hello, Priya!

# Pass as argument
def apply(func, value):
    return func(value)

print(apply(greet, "Vishal"))   # Hello, Vishal!
print(apply(len, [1,2,3]))      # 3
print(apply(str.upper, "hi"))   # HI

# Return from function
def get_operation(op):
    if op == "+": return lambda a, b: a + b
    if op == "*": return lambda a, b: a * b

add = get_operation("+")
print(add(3, 4))  # 7

### ✏️ Exercise 1 — Functions as Arguments 🔧

In [ ]:
# TODO: Write a function that applies a transformation to every item in a list
def transform(data, func):
    return [___ for item in data]

nums = [1, 2, 3, 4, 5]
print(transform(nums, lambda x: x ** 2))         # [1, 4, 9, 16, 25]
print(transform(nums, lambda x: x * 10))          # [10, 20, 30, 40, 50]
print(transform(["hello", "world"], str.upper))   # ['HELLO', 'WORLD']

*💡 Hint: `func(item)`.*

---

## 4.2 Closures — Functions That Remember 🧠

A **closure** is a function that **remembers** variables from the enclosing scope, even after that scope has finished.

In [ ]:
def make_multiplier(factor):
    """Returns a function that multiplies by 'factor'."""
    def multiply(x):
        return x * factor   # 'factor' is remembered!
    return multiply

double = make_multiplier(2)
triple = make_multiplier(3)

print(double(5))   # 10
print(triple(5))   # 15
print(double(10))  # 20

# The 'factor' variable lives on inside the closure
print(f"double remembers: {double.__closure__[0].cell_contents}")  # 2

In [ ]:
# Closure with mutable state
def make_counter(start=0):
    count = [start]  # list to allow mutation
    def counter():
        count[0] += 1
        return count[0]
    return counter

c = make_counter()
print(c(), c(), c())  # 1 2 3

c2 = make_counter(100)
print(c2(), c2())  # 101 102  — independent!

### ✏️ Exercise 2 — Build Closures 🧠

In [ ]:
# TODO 1: Create a closure that adds a prefix to strings
def make_prefixer(prefix):
    def add_prefix(text):
        return ___
    return add_prefix

add_mr = make_prefixer("Mr. ")
add_dr = make_prefixer("Dr. ")
print(add_mr("Vishal"))  # Mr. Vishal
print(add_dr("Priya"))   # Dr. Priya

# TODO 2: Create a running average closure
def make_averager():
    values = []
    def averager(new_value):
        values.append(new_value)
        return ___
    return averager

avg = make_averager()
print(avg(10))   # 10.0
print(avg(20))   # 15.0
print(avg(30))   # 20.0

*💡 Hint: `prefix + text`. `sum(values) / len(values)`.*

---

## 4.3 Decorators — The `@` Magic 🎩

A **decorator** is a function that **wraps** another function, adding behavior before/after it.

```python
@decorator
def func(): ...       # same as: func = decorator(func)
```

In [ ]:
import time

# A timer decorator
def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"  ⏱️ {func.__name__} took {elapsed:.4f}s")
        return result
    return wrapper

@timer
def slow_sum(n):
    return sum(range(n))

@timer
def slow_list(n):
    return [x**2 for x in range(n)]

slow_sum(1_000_000)
slow_list(100_000)

In [ ]:
# A debug/logging decorator
def debug(func):
    def wrapper(*args, **kwargs):
        args_str = ", ".join(repr(a) for a in args)
        kwargs_str = ", ".join(f"{k}={v!r}" for k, v in kwargs.items())
        all_args = ", ".join(filter(None, [args_str, kwargs_str]))
        print(f"  📞 {func.__name__}({all_args})")
        result = func(*args, **kwargs)
        print(f"  📤 → {result!r}")
        return result
    return wrapper

@debug
def add(a, b):
    return a + b

@debug
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

add(3, 4)
greet("Priya", greeting="Hi")

### ✏️ Exercise 3 — Write a `@count_calls` Decorator 📊

In [ ]:
# TODO: Decorator that counts how many times a function is called
def count_calls(func):
    def wrapper(*args, **kwargs):
        wrapper.calls += 1
        print(f"  📊 {func.__name__} called {wrapper.calls} time(s)")
        return func(*args, **kwargs)
    wrapper.calls = ___
    return wrapper

@count_calls
def say_hello(name):
    return f"Hello, {name}!"

say_hello("Anita")
say_hello("Rahul")
say_hello("Sneha")
print(f"Total calls: {say_hello.calls}")  # 3

*💡 Hint: `wrapper.calls = 0`.*

### ✏️ Exercise 4 — Write a `@retry` Decorator 🔄

In [ ]:
import random

# TODO: Decorator that retries a function up to max_retries times if it raises an exception
def retry(max_retries=3):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_retries + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"  ⚠️ Attempt {attempt}/{max_retries} failed: {e}")
                    if attempt == max_retries:
                        raise
        return wrapper
    return decorator

@retry(max_retries=5)
def flaky_api():
    """Simulates an unreliable API."""
    if random.random() < 0.7:  # 70% failure rate
        raise ConnectionError("Server unavailable")
    return {"status": "ok", "data": 42}

try:
    result = flaky_api()
    print(f"  ✅ Success: {result}")
except ConnectionError:
    print("  ❌ All retries failed!")

---
## 4.4 `@functools.wraps` — Preserving Metadata 📋

In [ ]:
from functools import wraps

# Without @wraps — metadata is LOST!
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def my_func():
    """My important docstring."""
    pass

print(f"Name: {my_func.__name__}")  # 'wrapper' — WRONG!
print(f"Doc: {my_func.__doc__}")    # None — LOST!

# With @wraps — metadata preserved!
def good_decorator(func):
    @wraps(func)                     # <-- add this!
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def my_func2():
    """My important docstring."""
    pass

print(f"\nName: {my_func2.__name__}")  # 'my_func2' ✅
print(f"Doc: {my_func2.__doc__}")      # preserved ✅

### ✏️ Exercise 5 — Write a `@validate_types` Decorator 🛡️

In [ ]:
from functools import wraps

def validate_positive(func):
    """Decorator that ensures all numeric arguments are positive."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        for i, arg in enumerate(args):
            if isinstance(arg, (int, float)) and arg < 0:
                raise ValueError(f"Argument {i} must be positive, got {arg}")
        for key, val in kwargs.items():
            if isinstance(val, (int, float)) and val < 0:
                raise ValueError(f"'{key}' must be positive, got {val}")
        return func(*args, **kwargs)
    return wrapper

@validate_positive
def calculate_area(width, height):
    """Calculate rectangle area."""
    return width * height

print(calculate_area(5, 3))        # 15
print(calculate_area.__name__)     # calculate_area (preserved!)
print(calculate_area.__doc__)      # Calculate rectangle area.

try:
    calculate_area(-5, 3)
except ValueError as e:
    print(f"❌ {e}")

### ✏️ Exercise 6 — Stacking Decorators 📚

In [ ]:
from functools import wraps
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"  ⏱️ {func.__name__}: {time.time()-start:.4f}s")
        return result
    return wrapper

def debug(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"  🔍 Calling {func.__name__}({args}, {kwargs})")
        result = func(*args, **kwargs)
        print(f"  📤 {func.__name__} → {result}")
        return result
    return wrapper

# Stacking: applied bottom-up!
@timer     # outer (runs first)
@debug     # inner (wraps the actual function)
def compute(n):
    return sum(i**2 for i in range(n))

# Same as: compute = timer(debug(compute))
compute(100_000)

### ✏️ Exercise 7 — Decorator with Arguments: `@slow_down(seconds)` ⏳

In [ ]:
import time
from functools import wraps

# TODO: Write a decorator factory that slows down function execution
def slow_down(seconds=1):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            print(f"  ⏳ Waiting {seconds}s before {func.__name__}...")
            time.sleep(seconds)
            return func(*args, **kwargs)
        return wrapper
    return decorator

@slow_down(0.5)
def greet(name):
    return f"Hello, {name}!"

print(greet("Priya"))

### ✏️ Exercise 8 — Write Your Own: `@cache_result` 📦

In [ ]:
from functools import wraps

def cache_result(func):
    """Simple memoization decorator."""
    cache = {}
    @wraps(func)
    def wrapper(*args):
        if args not in cache:
            cache[args] = func(*args)
            print(f"  💾 Computed: {func.__name__}{args} = {cache[args]}")
        else:
            print(f"  ⚡ Cached:   {func.__name__}{args} = {cache[args]}")
        return cache[args]
    wrapper.cache = cache  # expose cache for inspection
    return wrapper

@cache_result
def fibonacci(n):
    if n < 2: return n
    return fibonacci(n-1) + fibonacci(n-2)

print(f"fib(10) = {fibonacci(10)}")
print(f"\nCache has {len(fibonacci.cache)} entries")
print(f"fib(8) = {fibonacci(8)}")  # instant — cached!

---
## 4.5 `functools` — lru_cache, partial, reduce 🧰

In [ ]:
from functools import lru_cache

# lru_cache — built-in memoization (way better than our manual cache!)
@lru_cache(maxsize=128)
def fib(n):
    if n < 2: return n
    return fib(n-1) + fib(n-2)

print(f"fib(50) = {fib(50):,}")
print(f"Cache info: {fib.cache_info()}")
fib.cache_clear()  # clear cache

In [ ]:
from functools import partial

# partial — freeze some arguments
def power(base, exp):
    return base ** exp

square = partial(power, exp=2)
cube = partial(power, exp=3)

print(f"square(5) = {square(5)}")  # 25
print(f"cube(3) = {cube(3)}")      # 27

# Practical: pre-configured print
debug_print = partial(print, "[DEBUG]", flush=True)
debug_print("Something happened")
debug_print("Another thing")

In [ ]:
from functools import reduce

# reduce — fold a list into a single value
nums = [1, 2, 3, 4, 5]

product = reduce(lambda a, b: a * b, nums)
print(f"Product: {product}")  # 120

longest = reduce(lambda a, b: a if len(a) > len(b) else b,
                 ["hello", "wonderful", "world", "hi"])
print(f"Longest: {longest}")  # wonderful

# Flatten with reduce
nested = [[1,2], [3,4], [5,6]]
flat = reduce(lambda a, b: a + b, nested)
print(f"Flat: {flat}")  # [1,2,3,4,5,6]

### ✏️ Exercise 9 — `lru_cache` Performance Test 🚀

In [ ]:
import time
from functools import lru_cache

# Without cache
def fib_slow(n):
    if n < 2: return n
    return fib_slow(n-1) + fib_slow(n-2)

# With cache
@lru_cache(maxsize=None)
def fib_fast(n):
    if n < 2: return n
    return fib_fast(n-1) + fib_fast(n-2)

# Compare
n = 35
start = time.time()
r1 = fib_slow(n)
t1 = time.time() - start

start = time.time()
r2 = fib_fast(n)
t2 = time.time() - start

print(f"fib({n}) = {r1}")
print(f"Without cache: {t1:.4f}s")
print(f"With cache:    {t2:.6f}s")
print(f"Speedup:       {t1/t2:,.0f}x faster! 🚀")

### ✏️ Exercise 10 — `partial` & `reduce` Challenges 🧩

In [ ]:
from functools import partial, reduce

# TODO 1: Create a partial that converts string to int with base 2 (binary)
bin_to_int = partial(___, base=___)
print(bin_to_int("1010"))   # 10
print(bin_to_int("11111"))  # 31

# TODO 2: Use reduce to find the GCD of a list of numbers
import math
nums = [48, 36, 24, 12]
gcd = reduce(___, nums)
print(f"GCD of {nums} = {gcd}")  # 12

# TODO 3: Use reduce to build a sentence from words
words = ["Python", "is", "absolutely", "amazing"]
sentence = reduce(lambda a, b: ___, words)
print(sentence)  # Python is absolutely amazing

*💡 Hint: `partial(int, base=2)`. `reduce(math.gcd, nums)`. `a + " " + b`.*

---

## 4.6 Walrus Operator (`:=`) — Assign & Use in One Step 🦦

In [ ]:
# Without walrus
line = input("Enter text (or 'quit'): ") if False else "hello"

# With walrus — assign and test in one line
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Filter + transform in one pass
results = [squared for x in data if (squared := x**2) > 20]
print(f"Squares > 20: {results}")  # [25, 36, 49, 64, 81, 100]

# Avoid double computation
import re
text = "My phone is 9876543210 and email is test@mail.com"
if (match := re.search(r'\d{10}', text)):
    print(f"Found phone: {match.group()}")  # 9876543210

# In while loops
numbers = iter([3, 7, 2, 8, 1, 9])
while (n := next(numbers, None)) is not None:
    if n > 5:
        print(f"First > 5: {n}")  # 7
        break

### ✏️ Exercise 11 — Walrus Operator Practice 🦦

In [ ]:
# TODO 1: Use walrus to find items where len > 5 and show the length
words = ["hello", "magnificent", "world", "extraordinary", "hi", "python"]
long_words = [(w, length) for w in words if (length := len(w)) > 5]
print(f"Long words: {long_words}")

# TODO 2: Use walrus to avoid computing expensive function twice
import math
data = [2, 10, 100, 1000, 5000]
log_results = [(x, log_val) for x in data if (log_val := math.log10(x)) > 1.5]
print(f"Log > 1.5: {log_results}")

### ✏️ Exercise 12 — Write an `@authorize` Decorator 🔐

In [ ]:
from functools import wraps

def authorize(required_role):
    """Decorator that checks user role before executing."""
    def decorator(func):
        @wraps(func)
        def wrapper(user, *args, **kwargs):
            if user.get("role") != required_role:
                return f"❌ Access denied! Need '{required_role}' role."
            return func(user, *args, **kwargs)
        return wrapper
    return decorator

@authorize("admin")
def delete_user(user, target):
    return f"✅ {user['name']} deleted user '{target}'"

@authorize("editor")
def edit_post(user, post_id, content):
    return f"✅ {user['name']} edited post #{post_id}"

admin = {"name": "Anita", "role": "admin"}
editor = {"name": "Rahul", "role": "editor"}
viewer = {"name": "Sneha", "role": "viewer"}

print(delete_user(admin, "old_user"))   # ✅
print(delete_user(viewer, "someone"))   # ❌
print(edit_post(editor, 42, "Updated")) # ✅
print(edit_post(viewer, 42, "Nope"))    # ❌

### ✏️ Exercise 13 — Class Decorator 🏗️

In [ ]:
# Decorators can modify classes too!

def add_repr(cls):
    """Auto-add __repr__ to any class."""
    def __repr__(self):
        attrs = ", ".join(f"{k}={v!r}" for k, v in self.__dict__.items())
        return f"{cls.__name__}({attrs})"
    cls.__repr__ = __repr__
    return cls

@add_repr
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

@add_repr
class Car:
    def __init__(self, make, model):
        self.make = make
        self.model = model

print(Person("Priya", 28))       # Person(name='Priya', age=28)
print(Car("Tata", "Nexon"))      # Car(make='Tata', model='Nexon')

### ✏️ Exercise 14 — Predict the Output: Decorator Puzzles 🧩

In [ ]:
from functools import wraps

def bold(func):
    @wraps(func)
    def wrapper(*a, **kw): return f"<b>{func(*a, **kw)}</b>"
    return wrapper

def italic(func):
    @wraps(func)
    def wrapper(*a, **kw): return f"<i>{func(*a, **kw)}</i>"
    return wrapper

@bold
@italic
def greet(name): return f"Hello, {name}"

# Predict the output:
print(greet("Priya"))  # ___

*💡 Hint: Applied bottom-up: italic first, then bold wraps it → `<b><i>Hello, Priya</i></b>`.*

---
## 🎯 Exercise 15 — Mini-Project: Route Decorator (Mini Web Framework) 🌐

In [ ]:
from functools import wraps

class MiniApp:
    """A tiny web framework demonstrating decorator-based routing."""
    
    def __init__(self, name):
        self.name = name
        self.routes = {}
    
    def route(self, path):
        """Register a URL path to a handler function."""
        def decorator(func):
            self.routes[path] = func
            @wraps(func)
            def wrapper(*args, **kwargs):
                return func(*args, **kwargs)
            return wrapper
        return decorator
    
    def handle(self, path):
        """Simulate handling a request."""
        if path in self.routes:
            return self.routes[path]()
        return f"404 Not Found: {path}"
    
    def list_routes(self):
        print(f"\n🌐 {self.name} Routes:")
        for path, func in self.routes.items():
            print(f"  {path} → {func.__name__}()")

# === Build the app ===
app = MiniApp("MyWebApp")

@app.route("/")
def home():
    return "🏠 Welcome to the homepage!"

@app.route("/about")
def about():
    return "📖 About us: We love Python!"

@app.route("/contact")
def contact():
    return "📧 Email: hello@python.org"

@app.route("/api/data")
def api_data():
    return {"status": "ok", "items": [1, 2, 3]}

# === Test ===
app.list_routes()

print("\n📨 Simulating requests:")
for path in ["/", "/about", "/api/data", "/missing"]:
    response = app.handle(path)
    print(f"  GET {path:<15} → {response}")

---
## 🎊 Session 4 Complete!

| ✅ | Concept | Key Takeaway |
|---|---------|-------------|
| 🧱 | First-class functions | Assign, pass, return functions |
| 🧠 | Closures | Functions that remember their environment |
| 🎩 | Decorators | `@decorator` wraps functions with extra behavior |
| 📋 | `@functools.wraps` | Preserve function metadata |
| 📚 | Stacking | Multiple decorators, applied bottom-up |
| 🏭 | Decorator factories | `@decorator(args)` pattern |
| 🚀 | `lru_cache` | Built-in memoization |
| 🔧 | `partial` | Freeze function arguments |
| 📦 | `reduce` | Fold a sequence into one value |
| 🦦 | Walrus `:=` | Assign and use in one expression |

### 🚀 Next: Session 5 — Regular Expressions
---

## 📖 Solutions

<details>
<summary><strong>Click to reveal all solutions</strong></summary>

### Ex 1: `func(item)`
### Ex 2: `prefix + text`. `sum(values)/len(values)`.
### Ex 3: `wrapper.calls = 0`.
### Ex 10: `partial(int, base=2)`. `reduce(math.gcd, nums)`. `a + " " + b`.
### Ex 14: `<b><i>Hello, Priya</i></b>` (bottom-up).

</details>